# 復電資料前處理

# 1. 資料讀取

In [49]:
import pandas as pd
import os

DATA_DIR = "raw_data"

# 載入合併後的原始資料
df = pd.read_csv(os.path.join(DATA_DIR, "bpa_data_merged.csv"), low_memory=False)

# Check
df.head()

,Out Datetime,In Datetime,Name,Voltage (kV),Line Type,Gen Flag,Length (miles),Duration (minutes),Outage Type,Cause Dispatch,...,Cause,Responsible System,O&M District,Transmission Owner NERC TADS,Out Datetime (PPT),In Datetime (PPT),Kilovolt,Megawatt Loss 1/,OMS Outage ID,OARS Outage ID
0,01/01/1988 11:00,NaN,Ashe-Marion No 2 500kV line,500.0,L,T,224.01,still out,Plan,Normally Out,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,01/01/1988 11:00,NaN,Slatt tap to Ashe-Marion No 2 500kV line,500.0,T,,0.10,still out,Plan,Normally Out,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,01/20/1995 10:31,NaN,Satsop-Grays Harbor Public Development Authori...,230.0,L,,0.10,still out,Plan,Normally Out,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,04/03/1995 10:00,NaN,Bell-AVA Beacon No 2 115kV line,115.0,L,,6.20,still out,Plan,Normally Out,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,06/13/1996 10:20,NaN,Alvey-Fairview No 1 230kV line,230.0,L,T,97.46,still out,Plan,Normally Out,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# 2. 資料檢視

## 特徵欄位項目

In [50]:
# 各年份欄位一覽
for year, group in df.groupby("year"):
    cols = [c for c in group.dropna(axis=1, how="all").columns]
    print(f"{year}: {cols}")

2014: ['Out Datetime', 'In Datetime', 'Name', 'Voltage (kV)', 'Line Type', 'Gen Flag', 'Length (miles)', 'Duration (minutes)', 'Outage Type', 'Cause Dispatch', 'Cause Field', 'Responsible System Dispatch', 'Responsible System Field', 'District', 'Outage ID', 'year']
2015: ['Out Datetime', 'In Datetime', 'Name', 'Voltage (kV)', 'Line Type', 'Gen Flag', 'Length (miles)', 'Duration (minutes)', 'Outage Type', 'Cause Dispatch', 'Cause Field', 'Responsible System Dispatch', 'Responsible System Field', 'District', 'Outage ID', 'year']
2016: ['Out Datetime', 'In Datetime', 'Name', 'Voltage (kV)', 'Line Type', 'Gen Flag', 'Length (miles)', 'Duration (minutes)', 'Outage Type', 'District', 'Outage ID', 'year', 'Cause', 'Responsible System']
2017: ['Out Datetime', 'In Datetime', 'Name', 'Voltage (kV)', 'Line Type', 'Gen Flag', 'Length (miles)', 'Duration (minutes)', 'Outage Type', 'District', 'Outage ID', 'year', 'Cause', 'Responsible System']
2018: ['Out Datetime', 'In Datetime', 'Name', 'Voltage

In [51]:
print(df.columns.tolist())

['Out Datetime', 'In Datetime', 'Name', 'Voltage (kV)', 'Line Type', 'Gen Flag', 'Length (miles)', 'Duration (minutes)', 'Outage Type', 'Cause Dispatch', 'Cause Field', 'Responsible System Dispatch', 'Responsible System Field', 'District', 'Outage ID', 'year', 'Cause', 'Responsible System', 'O&M District', 'Transmission Owner NERC TADS', 'Out Datetime (PPT)', 'In Datetime (PPT)', 'Kilovolt', 'Megawatt Loss 1/', 'OMS Outage ID', 'OARS Outage ID']


## Duplicated Data

In [52]:
f"重複值 : {df.duplicated().sum()}"

'重複值 : 0'

# 3. Feature Selection

* 目標：將各年份欄位名稱不一致的欄位統一，並捨棄不需要的欄位

* 處理項目

    **A. Cause 欄合併（2014–2015）**

    `Cause Field` 優先，NaN 時 fallback 到 `Cause Dispatch`，合併後統一欄名為 `Cause`

    * `Cause Field` : 現場判斷(優先)
    * `Cause Dispatch` : 調度員判斷

    **B. Voltage 欄統一**
    `Kilovolt`（2026）合併進 `Voltage (kV)`

    **C. 捨棄不需要欄位**
    最終保留：`Name`、`Voltage (kV)`、`Line Type`、`Duration (minutes)`、`Outage Type`、`Cause`、`year`

* 特徵欄位說明

    | 欄位名稱 | 說明 | 保留 |
    |---|---|---|
    | `Out Datetime` | 停電開始時間 | ✗ (有停電持續時間之欄位，不須此項資料)|
    | `In Datetime` | 復電時間 | ✗  (有停電持續時間之欄位，不須此項資料)|
    | `Name` | 輸電線路名稱 | ✓ |
    | `Voltage (kV)` | 線路電壓（kV） | ✓ |
    | `Line Type` | 線路類型 | ✗ (在識別是否為主線路或分支線，不需此資料)|
    | `Gen Flag` | 發電旗標 | ✗ |
    | `Length (miles)` | 線路長度（英里） | ✗ |
    | `Duration (minutes)` | 停電持續時間（分鐘） | ✓ |
    | `Outage Type` | 停電類型（Auto / Plan 等） | ✓ (計畫性停電並非意外造成，，需保留本欄為，以便後續篩選) |
    | `Cause Dispatch` | 調度員判斷原因（2014–2015） | ✗（合併至 Cause）|
    | `Cause Field` | 現場確認原因（2014–2015） | ✗（合併至 Cause）|
    | `Responsible System Dispatch` | 調度員判斷責任系統（2014–2015） | ✗ |
    | `Responsible System Field` | 現場確認責任系統（2014–2015） | ✗ |
    | `District` | 地區（2014–2017） | ✗ |
    | `Outage ID` | 停電事件編號 | ✗ |
    | `year` | 資料年份 | ✓ |
    | `Cause` | 停電原因（2016+） | ✓ |
    | `Responsible System` | 責任系統（2016+） | ✗ |
    | `O&M District` | 地區（2018+） | ✗ |
    | `Transmission Owner NERC TADS` | 輸電業者（NERC TADS 格式） | ✗ |
    | `Out Datetime (PPT)` | 停電開始時間，PPT 時區（2022+） | ✗ |
    | `In Datetime (PPT)` | 復電時間，PPT 時區（2022+） | ✗ |
    | `Kilovolt` | 線路電壓（2026，改名欄位） | ✗（合併至 Voltage (kV)）|
    | `Megawatt Loss 1/` | 中斷電力（MW）（2026） | ✗ |
    | `OMS Outage ID` | OMS 系統停電編號（2026） | ✗ |
    | `OARS Outage ID` | OARS 系統停電編號（2026） | ✗ |

In [53]:
# Cause 欄合併（2014-2015）：Cause Field 優先，NaN 時 fallback 到 Cause Dispatch
df["Cause"] = df["Cause"].fillna(df["Cause Field"]).fillna(df["Cause Dispatch"])

# Voltage 欄統一（2026）：Kilovolt 合併進 Voltage (kV)
df["Voltage (kV)"] = df["Voltage (kV)"].fillna(df["Kilovolt"])

# 保留分析所需欄位
keep_cols = ["Name", "Voltage (kV)",  "Duration (minutes)", "Outage Type", "Cause", "year"]
df = df[keep_cols]

# Check
df.head()

,Name,Voltage (kV),Duration (minutes),Outage Type,Cause,year
0,Ashe-Marion No 2 500kV line,500.0,still out,Plan,Normally Out,2014
1,Slatt tap to Ashe-Marion No 2 500kV line,500.0,still out,Plan,Normally Out,2014
2,Satsop-Grays Harbor Public Development Authori...,230.0,still out,Plan,Normally Out,2014
3,Bell-AVA Beacon No 2 115kV line,115.0,still out,Plan,Normally Out,2014
4,Alvey-Fairview No 1 230kV line,230.0,still out,Plan,Normally Out,2014


# 4. 檢視資料內容

In [54]:
# Outage Type 值統計
print("Outage Type:")
print(df["Outage Type"].value_counts())

# Cause 值統計
print("\nCause:")
print(df["Cause"].value_counts())

Outage Type:
Outage Type
Plan    28906
Auto    24192
Name: count, dtype: int64

Cause:
Cause
Maintenance                   16369
Unknown                        7705
Lightning                      4399
Weather                        2690
Construction                   2611
                              ...  
TT Noise                         14
Overload                          5
Line or Bank charging             4
Out of step                       2
PSPS (Pub Safety Pwr Shut)        2
Name: count, Length: 64, dtype: int64


## 統計因素原因

In [55]:
# 印出所有 Cause 值
print(df["Cause"].value_counts().to_string())

Cause
Maintenance                   16369
Unknown                        7705
Lightning                      4399
Weather                        2690
Construction                   2611
Switching                      1916
Voltage Control                1873
Normally Out                   1764
Emergency                      1292
Urgent                         1161
Tree blown                      843
Wind                            809
Oper Plan/RTCA Reqd Action      705
Forced (Configuration)          704
Line Material Failure           638
Ice                             623
Fire                            553
Terminal Equipment Failure      509
Tree                            492
Tree Blown                      431
Contamination                   409
Proximity/Other                 396
Forced                          351
Bird or Animal                  350
Human Element                   284
Load Control                    278
Testing                         252
Bird droppings        

# 5. Data Cleaning

## 目標
從合併後的資料中，篩選出適合分析的停電事件

## 處理步驟

**A. 篩選自動停電（Auto）**

僅保留非計畫性的自動停電事件，排除計畫性維修停電（`Plan`）

**B. 排除 still out**

`Duration (minutes)` 為 `still out` 表示該停電事件在資料截止日仍未復電，無法取得完整復電時間，予以排除

**C. Duration 轉數值**

將 `Duration (minutes)` 欄轉為數值型態（`float`），非數值內容轉為 `NaN`

**D. 排除瞬間停電（duration ≤ 0）**

持續時間為 0 或負值者視為瞬間停電，不納入分析

**E. 排除 Duration 為 NaN**

轉型後仍為 `NaN` 者直接排除該筆資料

**F. Cause 標準化**

將原始 Cause 值對應至以下六個類別，其餘歸為 `Other`

| 標準類別 | 對應原始值 |
|---|---|
| `Foreign Trouble` | `Foreign Object`, `Vehicle`, `Aircraft`, `Machinery, Logging`, `Machinery, Construction`, `Machinery, Farming`, `Bird or Animal`, `Bird droppings`, `Bird Droppings` |
| `Unknown` | `Unknown`, `Not Reported` |
| `Lightning` | `Lightning` |
| `Tree Blown` | `Tree blown`, `Tree Blown`, `Tree`, `Tree cut`, `Tree Cut` |
| `Wind` | `Wind`, `Galloping Conductors` |
| `Weather` | `Weather`, `Ice`, `Dust`, `Smoke`, `Contamination`, `Fire` |
| `Other` | 其餘所有(如 : 人為因素、山崩、施工意外...等) |

In [56]:
# A. 篩選自動停電（Auto）
df = df[df["Outage Type"] == "Auto"].copy()

# B. 排除 still out
df = df[df["Duration (minutes)"] != "still out"]

# C. Duration 轉數值
# "coerce" : 遭於無法解析的值，使用 NaN 寫入
df["Duration (minutes)"] = pd.to_numeric(df["Duration (minutes)"], errors="coerce")

# D. 排除瞬間停電（duration ≤ 0）
df = df[df["Duration (minutes)"] > 0]

# E. 排除 Duration 為 NaN
df = df[df["Duration (minutes)"].notna()]

# F. Cause 標準化
CAUSE_MAP = {
    "Foreign Trouble": ["Foreign Object", "Vehicle", "Aircraft", "Machinery, Logging",
                        "Machinery, Construction", "Machinery, Farming", "Bird or Animal",
                        "Bird droppings", "Bird Droppings"],
    "Unknown":         ["Unknown", "Not Reported"],
    "Lightning":       ["Lightning"],
    "Tree Blown":      ["Tree blown", "Tree Blown", "Tree", "Tree cut", "Tree Cut"],
    "Wind":            ["Wind", "Galloping Conductors"],
    "Weather":         ["Weather", "Ice", "Dust", "Smoke", "Contamination", "Fire"],
}

# 將 CAUSE_MAP 反轉為 value -> key 的對應
cause_lookup = {v: k for k, vs in CAUSE_MAP.items() for v in vs}

# 對應標準類別，未對應者歸為 Other
df["Cause"] = df["Cause"].map(cause_lookup).fillna("Other")

# Check
df["Cause"].value_counts()

Cause
Other              2592
Weather            1754
Unknown            1738
Tree Blown         1637
Lightning           835
Wind                579
Foreign Trouble     397
Name: count, dtype: int64

#  6. Check Missing Data

In [57]:
print("缺失值：")
print(df.isnull().sum())

缺失值：
Name                  0
Voltage (kV)          0
Duration (minutes)    0
Outage Type           0
Cause                 0
year                  0
dtype: int64


# 存檔

In [58]:
output_dir = "data"
os.makedirs(output_dir, exist_ok=True)
df.to_csv(os.path.join(output_dir, "df_clean.csv"), index=False)